In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_spamhunter_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [5]:
spamhunter_dataset = pd.read_csv('../Dataset/SPAM-Hunter_full_dataset_messages.csv', delimiter='abcdexyagh')
spamhunter_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23824\246060161.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  spamhunter_dataset = pd.read_csv('../Dataset/SPAM-Hunter_full_dataset_messages.csv', delimiter='abcdexyagh')


,Messages
0,CASH BUY AND HOLD XXXXX SHARES OF ALPSMOTOR XX...
1,Last X hr Left to Activate your Godaddy XX Off...
2,Pongal Gift 1.Opkt be bill on your card on 11....
3,NSE BSE CASH PREMIUM BUY `` MOHITIND `` 58.75 ...
4,XX SPINS be wait for you when you SignUp at Lu...


In [6]:
# spamhunter_dataset = transform_spamhunter_dict(spamhunter_dataset)
# spamhunter_dataset.head()

In [7]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'spamhunter Dataset_'+'.csv')['URL'].to_list())

In [8]:
spamhunter_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'spamhunter Dataset_'+'.csv')['URL']
spamhunter_dataset['Message Len'] = [len(str(i)) for i in spamhunter_dataset['Messages']]
spamhunter_dataset.head()

,Messages,Extracted URL,Message Len
0,CASH BUY AND HOLD XXXXX SHARES OF ALPSMOTOR XX...,NaN,123
1,Last X hr Left to Activate your Godaddy XX Off...,NaN,137
2,Pongal Gift 1.Opkt be bill on your card on 11....,NaN,64
3,NSE BSE CASH PREMIUM BUY `` MOHITIND `` 58.75 ...,NaN,145
4,XX SPINS be wait for you when you SignUp at Lu...,NaN,125


In [9]:
spamhunter_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'spamhunter Websites Analysis'+'.csv')
# spamhunter_website_analysis_data = spamhunter_website_analysis_data.drop(columns=['ham', 'spam'])
spamhunter_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://www.kycpaytm.com/in.php,www.kycpaytm.com,0,0,-1,0
1,points.co.in,points.co.in,0,0,200,0
2,bit.ly/im3,bit.ly,126176,8761,200,1
3,https://bit.ly/3goHo9H,bit.ly,126176,8761,200,1
4,privee.com,privee.com,114,0,200,1


In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
spamhunter_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23824\3155401110.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  spamhunter_website_analysis_data.iloc[0][0]


'http://www.kycpaytm.com/in.php'

In [12]:
# for row in spamhunter_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = spamhunter_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(spamhunter_website_analysis_data['FQDN'])}
website_data = spamhunter_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
spamhunter_dataset['FQDN'] = fqdn
spamhunter_dataset['Website Size in KB'] = website_size
spamhunter_dataset['Website Textual Content Length'] = text_content_len
spamhunter_dataset['Status Code'] = status_code
spamhunter_dataset['Parked'] = parked

In [16]:
spamhunter_dataset = spamhunter_dataset.replace('', np.nan)
spamhunter_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23824\2646251056.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spamhunter_dataset = spamhunter_dataset.replace('', np.nan)


,Messages,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,CASH BUY AND HOLD XXXXX SHARES OF ALPSMOTOR XX...,NaN,123,NaN,NaN,NaN,NaN,NaN
1,Last X hr Left to Activate your Godaddy XX Off...,NaN,137,NaN,NaN,NaN,NaN,NaN
2,Pongal Gift 1.Opkt be bill on your card on 11....,NaN,64,NaN,NaN,NaN,NaN,NaN
3,NSE BSE CASH PREMIUM BUY `` MOHITIND `` 58.75 ...,NaN,145,NaN,NaN,NaN,NaN,NaN
4,XX SPINS be wait for you when you SignUp at Lu...,NaN,125,NaN,NaN,NaN,NaN,NaN


In [17]:
# Counter(spamhunter_dataset['FQDN'].to_list())
spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna()) & (spamhunter_dataset['FQDN'].isna())]

,Messages,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(spamhunter_dataset['FQDN'].to_list())
spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna()) & (spamhunter_dataset['FQDN'].notna())]

,Messages,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
5,Dear Mr. Frohlinger Your IndiGo PNR be TGUGQA ...,http//ty.vfap.co/7y8dq56sordownloadmobileapp,160,http.,NaN,NaN,NaN,NaN
7,Planning A Homeparty With Friends Over The Lon...,http//goo.gl/W2XJwfUsecodeJANFESTToAvailXXDisc...,198,http.,NaN,NaN,NaN,NaN
11,Get your Cab Pass start at just Rs XX Enjoy lo...,http//bit.ly/2yRMOWG,135,http.,NaN,NaN,NaN,NaN
14,Hola Su cuenta de whatsapp caducara para activ...,http//wwwwhatsapp.es/idextensionSaludosEquipod...,142,http.,NaN,NaN,NaN,NaN
16,Sonrie ¡ Solo hoy ENVIO GRATUITO en toda la we...,http//hawke.rsbluemondayNopublicidadSTPal+4475...,134,http.,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
25815,Lieber Kunde wir bitten Sie den Antrag für die...,http//spk.noreplydegerman.de/tkn=b105,149,http.,NaN,NaN,NaN,NaN
25818,REVENUE Due to the recent update on our websit...,http//revenue.ros,139,http.,NaN,NaN,NaN,NaN
25819,your Account Will be block Today update your P...,http//shrtco.de/fdEeqNISAeSurveillance,101,http.,NaN,NaN,NaN,NaN
25820,Estimado cliente A partir del 28/08/2022 no po...,http//liceoyobilo.cl/ess/sistema/home/19:20,164,http.,NaN,NaN,NaN,NaN


In [19]:
print(len(spamhunter_dataset))

25826


In [20]:
#messages with URL
print(len(spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna())]), len(spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna())])/len(spamhunter_dataset))

11123 0.4306900023232402


In [21]:
# #smish messages with URL
# len(spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna()) & (spamhunter_dataset['class']==1)])

In [22]:
# #smish messages with URL
# len(spamhunter_dataset[(spamhunter_dataset['Extracted URL'].notna()) & (spamhunter_dataset['class']==0)])

In [23]:
#unique FQDN
len(set(spamhunter_dataset[(spamhunter_dataset['FQDN'].notna())]['FQDN']))

849

In [24]:
only_unique_live_websites_data = spamhunter_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

144


In [25]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

31


In [26]:
spamhunter_dataset.to_csv('../Dataset/Refined_Spam_Hunter_Dataset.csv', index=None)